# The Web Scraping Playbook: A Comprehensive Guide to Data Extraction
*(With Football Analytics examples)*

## Overview
This notebook serves as a structured reference guide for modern web scraping methodologies. As web architectures become increasingly complex, extracting data efficiently requires a versatile toolkit—from parsing static HTML to intercepting dynamic network requests and deploying large-scale asynchronous crawlers.

Through practical, football-focused examples (such as extracting match stats, player profiles, and shot maps), this guide explores **what** each extraction technique is, **when** it is best applied based on the target site's architecture, and the **foundational Python code** required to execute it effectively.

Notebook first written: 02/25/2026<br>
Notebook updated written: 03/04/2026

---
## Index

- **Technique 1**: `requests` + `BeautifulSoup4` --> Static HTML Parsing
- **Technique 2**: Analyzing the Fetch/XHR Network Tab --> Network Interception & Reverse Engineering Internal APIs
- **Technique 3**: `Selenium` & `undetected_chromedriver` --> Dynamic Rendering & Browser Automation
- **Technique 4**: Official API
- **Technique 5**: `Scrapy`
- **Technique 6**: Data as a Service (DaaS) & Third-Party Providers

---
### Technique 1: Static HTML Parsing

**What it is?**

This is the foundational pillar of web scraping. It is the most traditional, lightweight, and resource-efficient way to extract data from the web.

To understand this technique, it helps to view it as a two-step process handled by two distinct libraries:
- **requests**: This acts as the messenger. It sends an HTTP GET request to the website's server, asking for a specific URL. The server responds by sending back the raw HTML text document
- **BeautifulSoup4**: Raw HTML is just a massive string of text. BeautifulSoup takes that string and parses it into a structured tree. This allow us to search for specific data using HTML tags, class names, or IDs.

**When to use it?**

We should reach for this technique when dealing with Server-Side Rendered (SSR) websites.

Websites can be build in 2 differents aechitectures:

1. ***SSR***: Is a technique that generates full HTML for web pages on the server for each request, delivering ready-to-display content to the user's browser. <br>
User --> HTTP GET request --> web server --> queries its database and injects that data into HTML templates --> send back a HTML document --> BeautifulSoup --> User
   
2. ***SPA/CSR*** (Single Page Application/Client-Side Rendering): Is an architecture where the server sends a minimal, mostly empty HTML skeleton along with a bundle of JavaScript files. The rendering work is offloaded to the user's browser, which must execute the JavaScript to fetch the actual data (usually via background API calls) and dynamically build the content on the screen.<br>
The Real Browser Flow:<br>
User --> HTTP GET request --> web server --> sends back an empty HTML skeleton + JS files --> browser executes the JS --> JS makes secondary requests to an API --> API returns JSON data --> JS builds the tables/text on the screen --> User.

**Examples**

- Classic data repositories: This is perfect for sites like FBref, Transfermarket or Wikipedia. Because the data is not constantly changing in the same URL.

**When to avoid it?**

- *SPAs*: If a site relies on JavaScript to fetch data after the initial page load (like the Sofascore dynamic shot maps we looked at earlier), this technique will fail. BeautifulSoup will just parse an empty skeleton because it cannot execute the JavaScript required to populate the data.<br>
The Python Scraper Flow (Why Technique 1 fails here):<br>
Python (requests) --> HTTP GET request --> web server --> sends back an empty HTML skeleton + JS files --> Python stops here! (It cannot run JS) --> BeautifulSoup parses the empty skeleton --> Scraper gets no data.
  
- *Aggressive Anti-Bot Protections*: Modern security firewalls (like Cloudflare or Datadome) often check if the client can execute JavaScript to prove it is a real browser. A simple requests call will usually get blocked immediately with a 403 Forbidden error or a CAPTCHA page.

**Code Example**:

In [43]:
# Step 1: Libraries
import requests
from bs4 import BeautifulSoup

In [45]:
# Step 2: the url to scrape
url = "https://www.scrapethissite.com/pages/simple/"

In [47]:
# Step 3: make the request
response = requests.get(url)

In [52]:
# parenthesis
# Status Code is something that tell us the status of our request
# Examples: 200 --> Ok | 404 --> couldn't find the page
response.status_code

200

In [ ]:
# step 4: check the full HTML code of the url that we have
#response.text

'<!doctype html>\n<html lang="en">\n  <head>\n    <meta charset="utf-8">\n    <title>Countries of the World: A Simple Example | Scrape This Site | A public sandbox for learning web scraping</title>\n    <link rel="icon" type="image/png" href="/static/images/scraper-icon.png" />\n\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <meta name="description" content="A single page that lists information about all the countries in the world. Good for those just get started with web scraping.">\n\n    <link href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.5/css/bootstrap.min.css" rel="stylesheet" integrity="sha256-MfvZlkHCEqatNoGiOXveE8FIwMzZg4W85qfrfIFBfYc= sha512-dTfge/zgoMYpP7QbHy4gWMEGsbsdZeCXz7irItjcC3sPUFtf0kuFbDz/ixG7ArTxmDjLXDmezHubeNikyKGVyQ==" crossorigin="anonymous">\n    <link href=\'https://fonts.googleapis.com/css?family=Lato:400,700\' rel=\'stylesheet\' type=\'text/css\'>\n    <link rel="stylesheet" type="text/css" href="/static/css/styles.css"

In [58]:
# step 5: use BeautifulSoup to parse the data
soup = BeautifulSoup(response.content, "html.parser")

In [62]:
# step 6: use different function of BeautifulSoup

# select_one(): llow us to use CSS selectors to extract the data
title = soup.select_one("h1").text.strip()
print(title)

Countries of the World: A Simple Example
                            250 items


In [41]:
# strip() is to delete the blank spaces at the begining and at the final
description = soup.select_one('p[class="lead"]').text.strip()
print(description)

A single page that lists information about all the countries in the world. Good for those just get started with web scraping.
                            Practice looking for patterns in the HTML that will allow you to extract information about each country. Then, build a simple web scraper that makes a request to this page, parses the HTML and prints out each country's name.


In [64]:
# select(): This is to select more that one html object. In specific, select() is going to return everything that match with our selector
country_names = soup.select("h3[class='country-name']") # how this is a list, we cannot use .strip, for that reason we do the following linecode
country_names = [x.text.strip() for x in country_names]
print(country_names)

['Andorra', 'United Arab Emirates', 'Afghanistan', 'Antigua and Barbuda', 'Anguilla', 'Albania', 'Armenia', 'Angola', 'Antarctica', 'Argentina', 'American Samoa', 'Austria', 'Australia', 'Aruba', 'Åland', 'Azerbaijan', 'Bosnia and Herzegovina', 'Barbados', 'Bangladesh', 'Belgium', 'Burkina Faso', 'Bulgaria', 'Bahrain', 'Burundi', 'Benin', 'Saint Barthélemy', 'Bermuda', 'Brunei', 'Bolivia', 'Bonaire', 'Brazil', 'Bahamas', 'Bhutan', 'Bouvet Island', 'Botswana', 'Belarus', 'Belize', 'Canada', 'Cocos [Keeling] Islands', 'Democratic Republic of the Congo', 'Central African Republic', 'Republic of the Congo', 'Switzerland', 'Ivory Coast', 'Cook Islands', 'Chile', 'Cameroon', 'China', 'Colombia', 'Costa Rica', 'Cuba', 'Cape Verde', 'Curacao', 'Christmas Island', 'Cyprus', 'Czech Republic', 'Germany', 'Djibouti', 'Denmark', 'Dominica', 'Dominican Republic', 'Algeria', 'Ecuador', 'Estonia', 'Egypt', 'Western Sahara', 'Eritrea', 'Spain', 'Ethiopia', 'Finland', 'Fiji', 'Falkland Islands', 'Micron

In [66]:
# wildcard: re usefull to select class that contains in their names an especific text
country_names = soup.select("h3[class*='-name']") # in this case, the wildcard search for classes that contains "-name"
country_names = [x.text.strip() for x in country_names]
print(country_names)

['Andorra', 'United Arab Emirates', 'Afghanistan', 'Antigua and Barbuda', 'Anguilla', 'Albania', 'Armenia', 'Angola', 'Antarctica', 'Argentina', 'American Samoa', 'Austria', 'Australia', 'Aruba', 'Åland', 'Azerbaijan', 'Bosnia and Herzegovina', 'Barbados', 'Bangladesh', 'Belgium', 'Burkina Faso', 'Bulgaria', 'Bahrain', 'Burundi', 'Benin', 'Saint Barthélemy', 'Bermuda', 'Brunei', 'Bolivia', 'Bonaire', 'Brazil', 'Bahamas', 'Bhutan', 'Bouvet Island', 'Botswana', 'Belarus', 'Belize', 'Canada', 'Cocos [Keeling] Islands', 'Democratic Republic of the Congo', 'Central African Republic', 'Republic of the Congo', 'Switzerland', 'Ivory Coast', 'Cook Islands', 'Chile', 'Cameroon', 'China', 'Colombia', 'Costa Rica', 'Cuba', 'Cape Verde', 'Curacao', 'Christmas Island', 'Cyprus', 'Czech Republic', 'Germany', 'Djibouti', 'Denmark', 'Dominica', 'Dominican Republic', 'Algeria', 'Ecuador', 'Estonia', 'Egypt', 'Western Sahara', 'Eritrea', 'Spain', 'Ethiopia', 'Finland', 'Fiji', 'Falkland Islands', 'Micron

In [68]:
# select_one() + :-soup-contains(): This is use full to extract within an especific class an especific text
chile = soup.select_one("h3:-soup-contains('Chile')").text
print(chile.strip())

Chile


---
### Technique 2: Network Interception & Reverse Engineering Internal APIs

**What it is?**

This is a technique for SPAs websites. This is how it works:

1) **The Initial Load**: I navigate to a website like Sofascore (sofascore.com) in my browser. Only during this very first visit, my browser makes a standard GET request and downloads an empty HTML skeleton along with the heavy JavaScript bundle.

2) **The SPA Action**: I search a specific match and click on the "Shotmap" tab. Because it is a Single Page Application (SPA), the page does not reload or ask for another HTML file. Instead, the already-loaded JavaScript detects my click and silently executes an internal function in the background.

3) **The Interception**: To see what that JavaScript is doing, I open the browser's Developer Tools (F12) and navigate to the Network > Fetch/XHR tab.

4) **Capturing the API**: While the Network tab is recording, I trigger the shotmap view. I can now see the hidden API URL (e.g., api.sofascore.com/...) that the JavaScript contacted to fetch the specific data for Match 123 - This only appears once I visit this part of the website.

5) **The Python Execution**: I copy this exact API URL (and its Headers). In my Python script, I use requests.get() to hit this URL directly, completely bypassing the browser. The server responds with the raw, structured data in JSON format instead of HTML

**Why do websites uses SPAs?**

SPAs are built for dynamic, live-updating information. In a live football match, events happen every few seconds. If the site used SSR, the server would have to build a brand new HTML page, and the user's screen would flash and reload every time a player took a shot. By using an SPA, the site keeps a background connection open, quietly fetching tiny JSON packets to update the screen instantly without interrupting the user.

**When to use it?**

1) **SPAs and Dynamic Dashboards**: like a live football score, an interactive shot map, or infinite scrolling
2) **When you want structured data instantly**: Instead of fighting with BeautifulSoup to extract text from nested tags, intercepting the API hands you a perfectly structured Python dictionary or JSON file. It is practically begging to be turned into a Pandas DataFrame. --> This only works if the website use SPA.
3) **Scale and Speed**: Because you are only requesting raw data and bypassing the UI rendering entirely, this method is incredibly fast and consumes very little bandwidth. --> This only works if the website use SPA.

**When to avoid it?**

- **If the website uses SRR**
- **Cryptographic Tokens & Extreme Security**: Some internal APIs are protected by dynamic security tokens that change every few seconds, or they require solving a JavaScript challenge to generate a valid request signature. If you cannot replicate the exact headers the server expects, it will reject your Python script with a 403 Forbidden or 401 Unauthorized error. In these cases, you are forced to use Technique 3 (Selenium).

**Code Example**:

In [10]:
# Step 1: Libraries
import json
import os
import requests
import pandas as pd

In [14]:
# Step 2: URL extrated from Fetch/XHR
url = "https://api.sofascore.com/api/v1/event/15452764/shotmap"

# And, header of the URL
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "*/*",
    "Origin": "https://www.sofascore.com",
    "Referer": "https://www.sofascore.com/"
}

In [16]:
# Step 3: Make the request
response = requests.get(url, headers=headers)

In [18]:
# Step 4: Transform the data
data = response.json()

# JSON of Sofascore brings a main key called "shotmap" that contains the shots
shots_list = data['shotmap']

# We use json_normalize to flatten the dictionary and create the DataFrame
df_shots = pd.json_normalize(shots_list)

In [20]:
# Step 5: visualize the dataframe
df_shots.head()

,isHome,shotType,situation,bodyPart,goalMouthLocation,xg,xgot,id,time,addedTime,...,draw.goal.x,draw.goal.y,goalkeeper.sofascoreId,player.sofascoreId,goalType,blockCoordinates.x,blockCoordinates.y,blockCoordinates.z,draw.block.x,draw.block.y
0,True,miss,assisted,right-foot,left,0.035119,0.000000,6778881,90,2.0,...,40.0,86.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,False,miss,set-piece,left-foot,high,0.007458,0.000000,6778872,86,NaN,...,45.0,13.9,yannsommer,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,True,miss,assisted,head,high-right,0.053791,0.000000,6778870,84,NaN,...,60.8,34.7,NaN,m_thuram,NaN,NaN,NaN,NaN,NaN,NaN
3,True,miss,corner,head,high-left,0.046141,0.000000,6778856,81,NaN,...,39.5,56.9,NaN,m_thuram,NaN,NaN,NaN,NaN,NaN,NaN
4,True,goal,corner,right-foot,low-centre,0.318085,0.185204,6778818,76,NaN,...,49.7,81.0,NaN,alessandrobastoni,regular,NaN,NaN,NaN,NaN,NaN


---
### Technique 3: Dynamic Rendering & Browser Automation

**What it is?**

Instead of just sending a lightweight HTTP request to a server, this technique involves lauching a fully functional, real web browser (like Google Chrome) and crontolling it programmatically via Python.

Specifically, it works like this:

1) **import undetected_chromedriver as uc**: This library locates the chromedriver.exe binary file on your PC. At the byte level, it modifies properties like navigator.webdriver from "true" to "undefined" to hide your bot identity.
2) **The Launch**: The Python script opens a physical Google Chrome browser instance, but it does so by opening a Remote Debugging Port (a local network port, usually 9222).
3) **Communication**: From this point on, your Python script and Chrome communicate via the WebDriver Protocol, sending JSON messages back and forth through that local port.
4) **The Command**: Your code executes driver.get('https://website.com/match123'). Python packages this command into a JSON payload and sends it to Chrome through the local port.
5) **The Execution**: Chrome obeys. It builds the HTTP GET requesta and bypasses the firewalls. (Note: Because the request originates from a real web browser rather than a standard Python script, it successfully bypasses WAFs like Cloudflare).
6) **The Processing**:
   
    6.1) **SSR Website**: The server receives the request and creates the fully populated HTML response. This follows the exact same flow as Technique 1.<br>
    6.2) **SPA Website**:

       - Your script tells Chrome to navigate to a URL (e.g., sofascore.com/match123).
       - Behind the scenes, the JavaScript executes and CSR (Client-Side Rendering) takes place. It is precisely thanks to this CSR process that the dynamic HTML is generated.
       - Because this rendering process takes a moment to fetch the API data and draw the elements, we must use WebDriverWait to pause our script.
       - Afterward, in our script, we call driver.page_source.
       - When you execute driver.page_source in Python, you are not downloading anything new from the internet. You are essentially telling Chrome: "Hey, look at your RAM at this exact millisecond. Take all that HTML that your JavaScript just manufactured and assembled like Lego bricks... convert it into plain text and hand it over to me."
       - Once we have that giant HTML string in Python, we can easily parse it with BeautifulSoup and transform it into a Pandas DataFrame.

8) **The Close**: To close the program and free up memory, we use driver.quit().

**When to use it?**

- **Heavily Protected Internal APIs (The SPA Fallback)**: When a site uses SPA, but you cannot use Technique 2 because the internal APIs are protected by dynamic cryptographic tokens, complex headers, or signatures that change constantly. Instead of spending days reverse-engineering the encryption, you use Selenium to let the browser's JavaScript engine do the hard work of fetching the JSON and building the "Lego" DOM in its RAM. You then just steal the final HTML.

- **Complex User Interactions**: When the data you need requires human-like interaction to even trigger the CSR. For example, scraping a football betting site where you must click specific dropdowns to select a league, click "Load More Matches", or scroll down repeatedly to trigger infinite loading.

- **Bypassing Aggressive Anti-Bots (WAFs)**: When sites use firewalls like Cloudflare or Datadome that block standard Python requests. Because undetected_chromedriver is a real, physical browser that executes JavaScript challenges natively, it easily passes the "Checking if you are human" tests.


**When to avoid it?**

- **When Technique 1 (SSR) or Technique 2 (Network Interception) works**: This should always be your last resort. Launching a full Chrome browser is incredibly slow and consumes massive amounts of RAM and CPU. If the data is already in the static HTML (SSR) or easily available as a clean JSON via an intercepted API, using Selenium is like using a bazooka to open a door.

- **Large-Scale Web Crawling**: If your goal is to scrape 10,000 player profiles quickly, Selenium will take hours, whereas API interception (Technique 2) would take minutes.

**Code Example** (SRR):

In [2]:
# Step 1: Libraries
import pandas as pd
import undetected_chromedriver as uc

In [15]:
# Step 2: Start up the chrome instance
# (We use 'uc' here to bypass FBref's WAF, not because we need to render JavaScript.)
driver = uc.Chrome(version_main=145) # version_main is the chrome version

In [21]:
# Step 3: Make the request and receive the pre-built HTML
driver.get("https://fbref.com/en/comps/8/2024-2025/stats/2024-2025-Champions-League-Stats")

In [23]:
# Step 4: to get the especific part we want of our HTML
df = pd.read_html(driver.page_source, attrs = {'id':"stats_squads_standard_for"})[0]
df.head()

C:\Users\PC\AppData\Local\Temp\ipykernel_23632\2206008909.py:2: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(driver.page_source, attrs = {'id':"stats_squads_standard_for"})[0]


Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
                Squad               # Pl                Age   
0         eng Arsenal                 24               25.8   
1     eng Aston Villa                 25               27.1   
2         it Atalanta                 24               27.7   
3  es Atlético Madrid                 23               28.5   
4        es Barcelona                 27               25.1   

  Unnamed: 3_level_0 Playing Time                    Performance      ...  \
                Poss           MP Starts   Min   90s         Gls Ast  ...   
0               50.9           14    154  1260  14.0          30  18  ...   
1               47.3           12    132  1080  12.0          22  18  ...   
2               53.3           10    110   900  10.0          22  19  ...   
3               47.4           10    110   930  10.3          22  17  ...   
4               61.3           14    154  1290  14.3          40  30  ...   

                          Per 90 Minutes                           
  G-PK PK PKatt CrdY CrdR            Gls   Ast   G+A  G-PK G+A-PK  
0   28  2     5   27    0           2.14  1.29  3.43  2.00   3.29  
1   21  1     2   19    0           1.83  1.50  3.33  1.75   3.25  
2   21  1     3   19    1           2.20  1.90  4.10  2.10   4.00  
3   22  0     0   18    1           2.13  1.65  3.77  2.13   3.77  
4   37  3     3   14    2           2.79  2.09  4.88  2.58   4.67  

[5 rows x 21 columns]

In [10]:
# Step 5: clean de df
df.columns = ['_'.join(col).strip() for col in df.columns.values]
df.head()

,Unnamed: 0_level_0_Squad,Unnamed: 1_level_0_# Pl,Unnamed: 2_level_0_Age,Unnamed: 3_level_0_Poss,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,...,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK
0,eng Arsenal,24,25.8,50.9,14,154,1260,14.0,30,18,...,28,2,5,27,0,2.14,1.29,3.43,2.00,3.29
1,eng Aston Villa,25,27.1,47.3,12,132,1080,12.0,22,18,...,21,1,2,19,0,1.83,1.50,3.33,1.75,3.25
2,it Atalanta,24,27.7,53.3,10,110,900,10.0,22,19,...,21,1,3,19,1,2.20,1.90,4.10,2.10,4.00
3,es Atlético Madrid,23,28.5,47.4,10,110,930,10.3,22,17,...,22,0,0,18,1,2.13,1.65,3.77,2.13,3.77
4,es Barcelona,27,25.1,61.3,14,154,1290,14.3,40,30,...,37,3,3,14,2,2.79,2.09,4.88,2.58,4.67


**Code Example** (SPA):

---
### Technique 4: Official API

**What it is?**

This technique involves using a website's Official API to get data directly, legally, and in a perfectly structured format.

But what exactly is an API?<br>
API stands for Application Programming Interface. Think of it using the classic restaurant analogy:

- The Database (Kitchen): This is where all the raw data (ingredients) is stored. You are not allowed to go in there.
- You (Customer): You want specific data (a meal).
- The API (Waiter): The API is the messenger. It gives you a menu (the Documentation) of exactly what you are allowed to ask for. You give the API your order, it goes to the kitchen, and it brings your data back on a silver platter (usually as a clean JSON file).

Instead of scraping (which is like sneaking into the kitchen to steal ingredients), you just ask the waiter. Many platforms even wrap their API into a dedicated Python library so you don't even have to write the HTTP requests yourself.

**When to use it?** (And how to know if a site has one)

You should use this technique always, provided the official API exists and gives you the data you need. It is the most stable, ethical, and fastest way to get data.

How to know if a website has an API:

- **Check the Footer**: Scroll to the very bottom of the website. Look for links that say "Developers", "API", "Open Data", or "For Programmers".
  
- **Google the Magic Words**: Search Google for "[Website Name] API documentation" or "[Website Name] Python package".

- **Look for GitHub Repositories**: Many companies (like StatsBomb or Spotify) host their official Python API wrappers on GitHub.

**When to avoid it?**

- **Paywalls & Expensive Tiers**: Many sports data providers (like Opta or Sportradar) have official APIs, but they cost thousands of dollars a month.

- **Rate Limits**: The API might have strict rules, like "You can only request 100 matches per day." If you need 10,000 matches for a Machine Learning model, the official API will block you, whereas Technique 2 (Network Interception) might not have those limits.

- **Missing Data**: Companies often show deep, granular data on their visual website to attract users, but they intentionally hide that same data from their free public API. If the website shows the X/Y shot coordinates but the official API only gives you the final score, you have to go back to scraping.

**Code Example**:

In [34]:
# !pip install statsbombpy

# Step 1: Import the official Python wrapper for the API
from statsbombpy import sb

In [36]:
# Official documentation: https://github.com/statsbomb/statsbombpy

---
### Technique 5: Scrapy

Note: Unlike the previous techniques, Scrapy is not designed to be executed inside a Jupyter Notebook. It requires a dedicated project structure and is best developed in an IDE like VS Code.

**What it is?**

While requests and BeautifulSoup are libraries, Scrapy is a complete, open-source web crawling framework.

Instead of downloading one page at a time (synchronous), Scrapy is completely asynchronous. It can fire off hundreds of requests simultaneously, managing the queue, handling retries, routing data through cleaning pipelines, and exporting it directly to a database without pausing. It doesn't just "scrape" a single URL; it "crawls" by following links from page to page like a spider on a web.

**When to use it?**

1) **Massive Scale**: When you need to scrape hundreds or thousands of pages.
  
2) **Deep Crawling (Link Following)**: When you don't have a neat list of URLs, and instead need a bot to start on a homepage, and follows links by links inside the website.

3) **Data Pipelines**: When your extraction requires built-in data engineering. Scrapy allows you to set up "Item Pipelines" that automatically clean the scraped strings and insert them directly into a differents databases as they are downloaded.

**When to avoid it?**

- **Simple, One-Off Tasks**: If you just need to grab the current league table once, setting up a Scrapy project with its rigid folder structure is massive overkill. Technique 1 or 2 is much better.

**Advice**:

For a comprehensive understanding of Scrapy, I highly recommend reading this book: [Web Scraping with Python](https://www.oreilly.com/library/view/web-scraping-with/9781098145347/)

---
### Technique 6: Data as a Service (DaaS) & Third-Party Providers

**What is it?**

Data as a Service (DaaS) is the ultimate shortcut: instead of building the data pipeline yourself, you pay a specialized company to do it for you.

These companies have massive teams of engineers whose sole job is to scrape, clean, standardize, and host data from thousands of sources. They deal with the WAFs, the broken SPAs, the IP proxies, and the server maintenance. Once the data is perfectly clean, they sell you access to it—usually via a highly reliable REST API, an AWS S3 bucket drop, or a direct database connection.

In the sports analytics world, companies like Opta (Stats Perform), Sportradar, or API-Sports act as DaaS providers. They aggregate live data from stadiums and broadcasts globally and deliver it as a premium service

**When to use it?**

- Production-Grade Reliability (SLAs)
- Cost-Benefit of Engineering Hours
- Etc.

**When to avoid it?**

- You want to save your money
- Others